# Install Libraries

In [ ]:
%pip install stable-baselines3[extra] gymnasium[atari,other] ale-py shimmy

# Freeway — DQN via Stable-Baselines3

## The Game
In Freeway a chicken at the bottom of the screen must cross 10 lanes of traffic to reach the top. The agent receives +1 for each successful crossing. Cars move horizontally and reset the chicken back one lane if they hit it.

- **Observation**: 84×84 grayscale pixel frame
- **Actions**: NOOP, UP, DOWN (3 total)
- **Reward**: +1 per successful crossing

## Why Freeway After Pong?
Freeway has simpler dynamics (move up, dodge cars) and the optimal strategy is easy to understand visually — making it a good second Atari step before tackling games with more complex strategies.

## Algorithm: DQN with CNN Policy (SB3)
Same SB3 DQN setup as Pong but with fewer training steps needed thanks to simpler dynamics.

# Create & Test Environment

In [ ]:
import gymnasium as gym
import ale_py
import numpy as np
import matplotlib.pyplot as plt
from stable_baselines3.common.atari_wrappers import AtariWrapper
from stable_baselines3.common.vec_env import DummyVecEnv, VecFrameStack

gym.register_envs(ale_py)

ENV_ID = 'ALE/Freeway-v5'
GAME   = 'freeway'

def make_env(seed=0):
    def _init():
        env = gym.make(ENV_ID, render_mode='rgb_array', frameskip=4)
        env = AtariWrapper(env, frame_skip=1, clip_reward=True, terminal_on_life_loss=False)
        env.reset(seed=seed)
        return env
    return _init

# Smoke test — random agent
raw_env = gym.make(ENV_ID, render_mode='rgb_array', frameskip=4)
obs, _ = raw_env.reset()
print('Raw observation shape:', obs.shape)
print('Action space:', raw_env.action_space)

frame = raw_env.render()
plt.figure(figsize=(3,4))
plt.imshow(frame)
plt.title('Freeway — initial frame')
plt.axis('off')
plt.show()
raw_env.close()

vec_env = VecFrameStack(DummyVecEnv([make_env(i) for i in range(4)]), n_stack=4)
print('Stacked obs shape:', vec_env.observation_space.shape)

# Train Agent

In [ ]:
import os
from stable_baselines3 import DQN
from stable_baselines3.common.callbacks import CheckpointCallback

TIMESTEPS        = 300_000
CHECKPOINT_FREQ  = TIMESTEPS // 4
CHECKPOINT_LABELS = ['Untrained', '25% (75k)', '50% (150k)', '75% (225k)', '100% (300k)']

os.makedirs(f'models/{GAME}', exist_ok=True)
os.makedirs(f'logs/{GAME}', exist_ok=True)

model = DQN(
    'CnnPolicy',
    vec_env,
    learning_rate=1e-4,
    buffer_size=100_000,
    learning_starts=10_000,
    batch_size=32,
    tau=1.0,
    gamma=0.99,
    train_freq=4,
    target_update_interval=1000,
    exploration_fraction=0.1,
    exploration_final_eps=0.01,
    tensorboard_log=f'./logs/{GAME}/',
    verbose=1,
)

# Save untrained model as step 0
model.save(f'models/{GAME}/{GAME}_step0')
print(f'Saved untrained model -> models/{GAME}/{GAME}_step0.zip')

checkpoint_cb = CheckpointCallback(
    save_freq=max(CHECKPOINT_FREQ // 4, 1),
    save_path=f'models/{GAME}/',
    name_prefix=f'{GAME}_step',
    verbose=1,
)

print(f'Training {GAME} for {TIMESTEPS:,} timesteps...')
model.learn(total_timesteps=TIMESTEPS, callback=checkpoint_cb, progress_bar=True)
model.save(f'models/{GAME}/{GAME}_final')
print(f'Final model saved -> models/{GAME}/{GAME}_final.zip')

# Evaluate & Visualize

In [ ]:
from stable_baselines3 import DQN as DQN_load

model = DQN_load.load(f'models/{GAME}/{GAME}_final')

eval_env = VecFrameStack(DummyVecEnv([make_env(99)]), n_stack=4)
N_EVAL = 5
all_rewards, all_frames = [], []

for ep in range(N_EVAL):
    obs = eval_env.reset()
    total_reward, ep_frames = 0, []
    while True:
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, done, info = eval_env.step(action)
        total_reward += reward[0]
        if ep == 0:
            ep_frames.append(eval_env.render())
        if done[0]:
            break
    all_rewards.append(total_reward)
    if ep == 0:
        all_frames = ep_frames

eval_env.close()
print(f'Evaluation (final model) — Mean: {np.mean(all_rewards):.2f}  Rewards: {all_rewards}')

In [ ]:
n_show = min(6, len(all_frames))
fig, axes = plt.subplots(1, n_show, figsize=(3*n_show, 3))
for i, ax in enumerate(axes):
    idx = int(i * len(all_frames) / n_show)
    ax.imshow(all_frames[idx])
    ax.set_title(f'Step {idx}')
    ax.axis('off')
plt.suptitle('Freeway — DQN Greedy Policy')
plt.tight_layout()
plt.show()

In [ ]:
# %load_ext tensorboard
# %tensorboard --logdir logs/freeway

# Compare: Untrained vs Intermediate vs Fully Trained

We load each saved checkpoint and run 3 evaluation episodes, then compare mean reward and display a mid-episode frame from each stage.

In [ ]:
import glob, re

def eval_sb3_model(path, n_episodes=3):
    m = DQN_load.load(path)
    env_e = VecFrameStack(DummyVecEnv([make_env(42)]), n_stack=4)
    rewards, frames = [], []
    for ep in range(n_episodes):
        obs = env_e.reset()
        total, ep_frames = 0, []
        while True:
            action, _ = m.predict(obs, deterministic=True)
            obs, reward, done, _ = env_e.step(action)
            total += reward[0]
            if ep == 0:
                ep_frames.append(env_e.render())
            if done[0]:
                break
        rewards.append(total)
        if ep == 0:
            frames = ep_frames
    env_e.close()
    return rewards, frames

ckpt_files = sorted(
    glob.glob(f'models/{GAME}/{GAME}_step*.zip'),
    key=lambda p: int(re.search(r'step(\d+)', p).group(1))
)
final_file = f'models/{GAME}/{GAME}_final.zip'
all_paths  = ckpt_files + [final_file]

def make_label(path):
    m = re.search(r'step(\d+)', path)
    if m:
        step = int(m.group(1))
        pct  = step / TIMESTEPS * 100
        return f'step {step:,} ({pct:.0f}%)'
    return 'Final (100%)'

labels = [make_label(p) for p in all_paths]
print('Checkpoints to evaluate:')
for lbl, p in zip(labels, all_paths):
    print(f'  {lbl:25s}  {p}')

In [ ]:
ck_rewards, ck_frames = [], []
print(f'{"Checkpoint":28s}  Mean   Rewards')
print('-' * 60)
for path, label in zip(all_paths, labels):
    rewards, frames = eval_sb3_model(path)
    ck_rewards.append(rewards)
    ck_frames.append(frames)
    print(f'{label:28s}  {np.mean(rewards):5.2f}  {rewards}')

In [ ]:
colors_ck = plt.cm.RdYlGn(np.linspace(0.1, 0.9, len(labels)))
means = [np.mean(r) for r in ck_rewards]

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(labels, means, color=colors_ck, edgecolor='black', linewidth=0.8)
for bar, m in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{m:.2f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_ylabel('Mean Reward (3 episodes)')
ax.set_title('Freeway DQN — Mean Reward per Checkpoint')
ax.grid(axis='y', alpha=0.3)
plt.setp(ax.get_xticklabels(), rotation=20, ha='right')
plt.tight_layout()
plt.show()

# Frame strip
fig2, axes2 = plt.subplots(1, len(labels), figsize=(3*len(labels), 3))
for ax, frames, label in zip(axes2, ck_frames, labels):
    if frames:
        ax.imshow(frames[len(frames) // 2])
    ax.set_title(label, fontsize=8)
    ax.axis('off')
plt.suptitle('Freeway — Mid-episode Frame at Each Checkpoint', fontsize=11)
plt.tight_layout()
plt.show()